# PyArrow: typed columns and efficient data files

A sales dataset must move between tools without losing types, missingness or monetary meaning. Define the schema, compute a report, then write and selectively read a columnar dataset. Check both the logical values and the physical representation at each handoff.

Apache Arrow defines a columnar memory format for exchanging data between tools. PyArrow exposes arrays, tables, compute functions, and file readers and writers in Python.

We will build a small invented sales table, preserve missing values and types, write Parquet and Arrow IPC files, and scan a partitioned dataset. All examples run offline after installation. Run cells in order.

## 1. Install and import

If needed, run `%pip install pyarrow pandas` in a separate cell.

| Object or format | Purpose |
| --- | --- |
| Array | One typed sequence of values |
| ChunkedArray | One logical column backed by multiple arrays |
| RecordBatch | A set of equal-length arrays with a schema |
| Table | Named columns that may contain multiple chunks |
| Parquet | Compressed columnar storage on disk |
| Arrow IPC | Serialization designed around Arrow's memory representation |

Arrow and Parquet are related technologies with different roles; they are not interchangeable names.

In [ ]:
from pathlib import Path
from datetime import date
from decimal import Decimal
import json
import tempfile
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow.csv as pacsv
import pyarrow.ipc as ipc

print("PyArrow:", pa.__version__)
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)

## 2. Create typed arrays with nulls

An Arrow array has one data type. Nulls represent absent values independently from ordinary numerical values such as zero. Explicit types make the intended schema visible instead of relying entirely on inference.

In [ ]:
quantities = pa.array([2, 1, None, 3, 2, 4], type=pa.int32())
print("Type:", quantities.type)
print("Values:", quantities.to_pylist())
print("Null count:", quantities.null_count)
print("Validity:", pc.is_valid(quantities).to_pylist())

## 3. Make the data contract explicit before exchanging it

Predict which fields can be missing and which require a fixed type. Compare the constructed schema with those expectations. A file that loads successfully can still encode a price or identifier in an unsuitable type.

Each row represents one sale line. Prices use decimal128 with precision 8 and scale 2: up to eight total decimal digits, two after the decimal point. Dates use date32 rather than strings.

The missing quantity is intentional. Monetary values are generated teaching data, and all prices use the same currency, USD.

In [ ]:
schema = pa.schema([
    ("sale_id", pa.int64()), ("region", pa.string()), ("sale_date", pa.date32()),
    ("quantity", pa.int32()), ("unit_price", pa.decimal128(8, 2)),
])
table = pa.Table.from_pydict({
    "sale_id": [1, 2, 3, 4, 5, 6],
    "region": ["East", "West", "East", "West", "East", "West"],
    "sale_date": [date(2025, 1, day) for day in [1, 1, 2, 2, 3, 3]],
    "quantity": quantities,
    "unit_price": [Decimal(value) for value in ["3.50", "5.00", "4.00", "2.50", "6.00", "1.25"]],
}, schema=schema)
print(table.schema)
print("Rows:", table.num_rows, "Columns:", table.num_columns)
print(table.to_pylist())

## 4. Select and slice

Selecting columns projects a table to the fields you need. Slicing uses an offset and length, not a stop index. Many Arrow slicing operations can share underlying buffers instead of copying all data.

Arrow arrays are logically immutable. Operations return new objects; they do not update the original table in place.

In [ ]:
subset = table.select(["sale_id", "quantity"]).slice(1, 3)
print(subset.to_pylist())
print("Original row count:", table.num_rows)

## 5. Understand null behavior in computations

A comparison with a null produces a null Boolean. Filtering drops rows with null selection values by default. Filling a missing quantity with zero is a business decision, not a neutral cleanup.

We demonstrate filling in a separate array while retaining missingness in the original sales table.

In [ ]:
large_quantity = pc.greater_equal(table["quantity"], 2)
filtered = table.filter(large_quantity)
filled_quantities = pc.fill_null(table["quantity"], 0)
print("Comparison mask:", large_quantity.to_pylist())
print("Selected IDs:", filtered["sale_id"].to_pylist())
print("Filled copy:", filled_quantities.to_pylist())
print("Original missing values:", table["quantity"].null_count)

## 6. Preserve the intended arithmetic for money

Calculate one revenue value by hand before running the column expression. Check its type and scale as well as its displayed value. Formatting a floating-point number to two decimals is not the same as preserving decimal arithmetic.

Multiplying quantity by unit price produces decimal revenue. Null quantity propagates to null revenue. This distinguishes unknown revenue from a known zero.

The compute kernel can widen decimal precision during arithmetic. Inspect the result type instead of assuming it matches the input price type.

In [ ]:
revenue = pc.multiply(table["quantity"], table["unit_price"])
enriched = table.append_column("revenue", revenue)
print("Revenue type:", revenue.type)
print("Revenue values:", revenue.to_pylist())
print("Original columns:", table.column_names)

## 7. Group and sort

Group by region and sum known revenue. The default sum skips nulls, so the East total is a partial total. Count non-null revenues alongside row counts to make incomplete data visible.

Aggregation order is not a business ordering guarantee. Sort explicitly for deterministic presentation.

In [ ]:
summary = enriched.group_by("region").aggregate([
    ("sale_id", "count"), ("revenue", "count"), ("revenue", "sum")
]).sort_by([("region", "ascending")])
print(summary.to_pylist())

## 8. Chunks and record batches

A logical column can contain multiple chunks. Concatenating compatible tables can preserve those chunks. `combine_chunks` may allocate memory to consolidate them, so use it when a consumer needs contiguous data.

Record batches are useful for processing in pieces. Converting this already-loaded table to batches illustrates the API; it does not undo the memory used to create the table.

In [ ]:
combined = pa.concat_tables([table.slice(0, 3), table.slice(3, 3)])
print("Quantity chunks:", combined["quantity"].num_chunks)
compact = combined.combine_chunks()
print("After consolidation:", compact["quantity"].num_chunks)
batches = combined.to_batches(max_chunksize=2)
print("Batch row counts:", [batch.num_rows for batch in batches])
print("Reconstructed table equals original:", pa.Table.from_batches(batches).equals(table))

## 9. Interoperate with Pandas

Conversions may allocate memory; zero-copy conversion depends on types, nulls, and layout. Pandas Arrow-backed dtypes can retain nullable integer and decimal representation.

`preserve_index=False` avoids adding a DataFrame index as a data column. Supply the expected schema when converting back to enforce the intended types.

In [ ]:
frame = table.to_pandas(types_mapper=pd.ArrowDtype)
print(frame.dtypes)
roundtrip_table = pa.Table.from_pandas(frame, schema=schema, preserve_index=False)
print("Pandas round-trip values match:", roundtrip_table.equals(table, check_metadata=False))

## 10. Check a file round trip rather than assuming it is lossless

After reading the Parquet file, compare schema, row count, nulls and values with the original table. Decide which metadata is also required by downstream consumers.

Parquet groups rows into row groups and stores column data with metadata. Reading only selected columns avoids materializing unneeded fields. Row-group size is a tuning choice: three rows is intentionally tiny for this demonstration, not a production recommendation.

This cell replaces the named file when rerun. We use Snappy if the installed build supports it, otherwise uncompressed storage.

In [ ]:
parquet_path = output_dir / "pyarrow_sales.parquet"
compression = "snappy" if pa.Codec.is_available("snappy") else "NONE"
pq.write_table(enriched, parquet_path, compression=compression, row_group_size=3)
parquet_file = pq.ParquetFile(parquet_path)
print("Row groups:", parquet_file.metadata.num_row_groups)
print("Stored rows:", parquet_file.metadata.num_rows)
projected = pq.read_table(parquet_path, columns=["sale_id", "region", "revenue"])
restored = pq.read_table(parquet_path)
print(projected.to_pylist())

## 11. Write a partitioned dataset

Hive-style partition directories encode values in names such as `region=East`. The partition field can be reconstructed from the path when reading.

We create a unique output directory per run to avoid stale files or duplicates from previous runs. Partitioning by a small number of useful categories can help pruning; partitioning by nearly unique IDs can create many tiny files.

In [ ]:
dataset_dir = Path(tempfile.mkdtemp(prefix="pyarrow_regions_", dir=output_dir))
partitioning = ds.partitioning(pa.schema([("region", pa.string())]), flavor="hive")
ds.write_dataset(enriched, base_dir=dataset_dir, format="parquet", partitioning=partitioning,
                 basename_template="part-{i}.parquet", existing_data_behavior="error")
dataset = ds.dataset(dataset_dir, format="parquet", partitioning=partitioning)
print("Dataset directory:", dataset_dir)
print("Files:", sorted(str(path.relative_to(dataset_dir)) for path in dataset_dir.rglob("*.parquet")))

## 12. Read only the slice needed for the report

State the required columns and filter before building the scanner. Verify that the collected rows satisfy the filter. A smaller result does not by itself prove all irrelevant bytes were skipped; file layout and metadata affect the physical work.

Dataset expressions describe a query. Projection chooses columns; a filter chooses rows. The scanner can use partition information and Parquet statistics to skip irrelevant data where possible.

This example needs only East rows with known quantities of at least two. Use `&` and parentheses for expression conjunction, not Python's `and`. Dataset row order should not be assumed; sort when needed.

In [ ]:
predicate = (ds.field("region") == "East") & (ds.field("quantity") >= 2)
scan = dataset.scanner(columns=["sale_id", "region", "revenue"], filter=predicate, batch_size=2)
scanned = scan.to_table().sort_by([("sale_id", "ascending")])
print(scanned.to_pylist())
matching_fragments = list(dataset.get_fragments(filter=ds.field("region") == "East"))
print("East partition fragments:", len(matching_fragments))

## 13. Process scanner batches incrementally

Iterating over batches avoids collecting the full scan result in one Python table. Batch size limits output batch rows, but it is not a strict cap on all reader memory: buffering, decompression, and readahead also use memory.

Sum only known revenues and count missing ones separately. This prevents a partial total from being reported as fully observed revenue.

In [ ]:
known_total = Decimal("0.00")
missing_revenues = 0
scanned_rows = 0
for batch in dataset.scanner(columns=["revenue"], batch_size=2).to_batches():
    values = batch.column(0)
    subtotal = pc.sum(values).as_py()
    if subtotal is not None:
        known_total += subtotal
    missing_revenues += values.null_count
    scanned_rows += batch.num_rows
print("Rows scanned:", scanned_rows)
print("Known revenue total:", known_total)
print("Rows with unknown revenue:", missing_revenues)

## 14. Serialize Arrow IPC

An IPC file preserves an Arrow schema and its record batches. Use matching file writer/reader APIs. The stream variant has a different framing and corresponding stream APIs.

Context managers close file resources. This example reads the whole IPC table back for comparison.

In [ ]:
ipc_path = output_dir / "pyarrow_sales.arrow"
with pa.OSFile(str(ipc_path), "wb") as sink:
    with ipc.new_file(sink, enriched.schema) as writer:
        writer.write_table(enriched)
with pa.OSFile(str(ipc_path), "rb") as source:
    ipc_restored = ipc.open_file(source).read_all()
print("IPC table matches:", ipc_restored.equals(enriched))

## 15. Export the summary with its limitations

Parquet preserves the typed summary. CSV is convenient for inspection, but does not store an Arrow schema. A JSON manifest records row counts, missingness, and output locations.

Decimal values are represented as strings in JSON to preserve their exact textual value. The known-revenue total intentionally excludes the row with unknown quantity. These named files are replaced when rerun.

In [ ]:
summary_path = output_dir / "pyarrow_region_summary.parquet"
csv_path = output_dir / "pyarrow_region_summary.csv"
pq.write_table(summary, summary_path)
pacsv.write_csv(summary, csv_path)
manifest = {"source_rows": table.num_rows, "known_revenue_total_usd": str(known_total),
            "unknown_revenue_rows": missing_revenues, "partitioned_dataset": str(dataset_dir),
            "parquet_file": str(parquet_path), "ipc_file": str(ipc_path)}
(output_dir / "pyarrow_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(summary.to_pylist())
print("Saved Parquet, CSV, IPC, partitioned files, and a JSON manifest.")

## 16. Practice and troubleshooting

1. Add a third region and inspect the resulting partition directories.
2. Compare filtering unknown quantities out with explicitly filling them as zero. Explain the change in meaning.
3. Scan only West rows and verify their known revenue independently.
4. Read the CSV back with an explicit conversion schema and compare it with the typed summary.

| Symptom | What to check |
| --- | --- |
| Decimal conversion fails | Use Decimal values and sufficient precision and scale |
| Filter removes a missing row | Null selection values are dropped by default |
| Dataset rows appear reordered | Sort explicitly; file and parallel scan order are not a contract |
| Partition column is absent | Supply the correct partitioning configuration when reading |
| Repeated writes produce duplicate data | Choose a deliberate overwrite/append policy or a fresh directory |
| Converting to Pandas changes nullable types | Use appropriate nullable or Arrow-backed dtypes |
| A scan still uses substantial memory | Projection, batch size, buffering, and readahead all matter |

See the official [data model](https://arrow.apache.org/docs/python/data.html), [compute functions](https://arrow.apache.org/docs/python/compute.html), [Parquet guide](https://arrow.apache.org/docs/python/parquet.html), and [dataset guide](https://arrow.apache.org/docs/python/dataset.html).

## Explain what the evidence supports

Explain one type that would be unsafe to infer casually. Verify a round trip and describe how column selection, filtering and batching affect the amount of data materialized.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: publish a typed payment dataset and scan only needed columns

A payment export contains retries, invalid amounts, malformed dates and unknown customer keys. Refunds are signed negative amounts, so deleting every negative amount would inflate net receipts. Reconcile 6,060 raw rows into 60 retries, 50 rejected events and 5,950 usable events. These are generated operational-style records, not actual customer payments. The generator's clean source is used only as a verification oracle, never to fill invalid source fields. Work within a small laptop-sized budget, then identify which intermediate tables or buffers would grow at scale.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the schema at the ingestion boundary

An Arrow table can efficiently preserve types, but it cannot decide that the string `bad` should mean zero. Quarantine malformed source values before constructing a typed table. Use integer cents here because all source amounts are whole cents. A successful Parquet write should preserve row count, null policy and signed totals.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(91)
p2_n=6000
p2_source=pd.DataFrame({'event_id':np.arange(p2_n),'customer_id':p2_rng.integers(1,121,p2_n),
    'day':pd.Timestamp('2025-01-01')+pd.to_timedelta(p2_rng.integers(0,90,p2_n),unit='D'),
    'amount_cents':p2_rng.integers(100,100000,p2_n)})
p2_source.loc[p2_source.event_id%17==0,'amount_cents'] *= -1 # Signed refunds are valid.
p2_raw=p2_source.astype(str)
p2_raw.loc[0:19,'amount_cents']='bad'
p2_raw.loc[20:29,'amount_cents']=''
p2_raw.loc[30:37,'day']='not-a-date'
p2_raw.loc[38:49,'customer_id']='9999'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-60:]],ignore_index=True)
p2_customers=pd.DataFrame({'customer_id':np.arange(1,121),'region':np.tile(['East','West','North','South'],30)})
p2_raw.to_csv(p2_out/'ledger_raw.csv',index=False)
# Independent generator-based expected population; never repair a bad value from this oracle.
p2_expected=p2_source.iloc[50:]
p2_expected_total=int(p2_expected.amount_cents.sum())
assert len(p2_raw)==6060 and len(p2_expected)==5950

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
p2_unique=p2_raw.drop_duplicates().copy()
p2_unique['cents']=pd.to_numeric(p2_unique.amount_cents,errors='coerce')
p2_unique['date']=pd.to_datetime(p2_unique.day,errors='coerce')
p2_unique['customer_id']=pd.to_numeric(p2_unique.customer_id,errors='coerce')
p2_ok=p2_unique.cents.notna()&p2_unique.date.notna()&p2_unique.customer_id.isin(p2_customers.customer_id)
p2_clean=p2_unique[p2_ok].copy();p2_clean['cents']=p2_clean.cents.astype('int64')
p2_typed=pa.table({'event_id':pa.array(p2_clean.event_id.astype('int64'),type=pa.int64()),
 'customer_id':pa.array(p2_clean.customer_id,type=pa.int64()),'date':pa.array(p2_clean.date.dt.date,type=pa.date32()),
 'cents':pa.array(p2_clean.cents,type=pa.int64())})
assert p2_typed.num_rows==5950 and int(pc.sum(p2_typed['cents']).as_py())==p2_expected_total
pq.write_table(p2_typed,p2_out/'arrow_ledger.parquet',row_group_size=1000,compression='zstd')
print(pq.ParquetFile(p2_out/'arrow_ledger.parquet').metadata.num_row_groups,'row groups')

### Work through full materialization versus record batches

Both approaches calculate January net receipts. The first reads the whole table before filtering; the second asks a scanner for only `cents` under a date predicate and reduces batches. This bounds Python-side output accumulation. Row-group pruning depends on metadata and ordering; a random-date file may offer little skipping even with a correct filter.

In [ ]:
from datetime import date
p2_t=time.perf_counter();p2_full=pq.read_table(p2_out/'arrow_ledger.parquet')
p2_mask=pc.less(p2_full['date'],pa.scalar(date(2025,2,1),type=pa.date32()))
p2_january=pc.sum(p2_full.filter(p2_mask)['cents']).as_py();p2_full_time=time.perf_counter()-p2_t
p2_dataset=ds.dataset(p2_out/'arrow_ledger.parquet',format='parquet')
p2_t=time.perf_counter();p2_total=0;p2_rows=0;p2_largest=0
for p2_batch in p2_dataset.scanner(columns=['cents'],filter=ds.field('date')<date(2025,2,1),batch_size=256).to_batches():
    p2_total+=pc.sum(p2_batch.column('cents')).as_py() or 0
    p2_rows+=p2_batch.num_rows;p2_largest=max(p2_largest,p2_batch.nbytes)
p2_scan_time=time.perf_counter()-p2_t
p2_oracle=p2_expected[p2_expected.day<'2025-02-01']
assert p2_total==p2_january==int(p2_oracle.amount_cents.sum()) and p2_rows==len(p2_oracle)
p2_report=pd.DataFrame([{'method':'full','seconds':p2_full_time,'materialized_bytes':p2_full.nbytes},
 {'method':'scanner','seconds':p2_scan_time,'materialized_bytes':p2_largest}])
display(p2_report);p2_report.to_csv(p2_out/'arrow_scan_comparison.csv',index=False)

### Interpret the batch measurement

`largest batch bytes` describes the largest output batch, not the process's entire peak memory. File decompression, buffering and the original generated fixture still cost memory. The exact signed total and row count show both paths answered the same question.

**Try:** append a second Parquet file with cents stored as strings. **Worked reasoning:** an explicit dataset schema can expose or control the mismatch, but a requested cast can still fail on malformed text. Preserve ingestion quarantine rather than relying on file concatenation to repair inconsistent schemas. Sorting a large file by date before writing may improve pruning, at the cost of the sort itself.